<a href="https://colab.research.google.com/github/majedabdalla/youtube/blob/main/YouTube_Shorts_Daily_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 YouTube Shorts Daily Automation Pipeline
### AI-powered content factory — topic → finished Short in one run

---

## What this notebook does
Generates a complete **9:16 vertical YouTube Short** from a single topic prompt using:
- **Gemini** — script, scene plan, metadata
- **edge-tts** — high-quality neural narration
- **ModelScope / SDXL-Turbo** — AI video clips or animated images (Ken Burns)
- **Whisper** — burned-in subtitles
- **MoviePy + ffmpeg** — assembly, transitions, final render
- **YouTube Data API v3** — optional direct upload

## What you must set before running
| Setting | Cell | Description |
|---------|------|-------------|
| `TOPIC_PROMPT` | Cell 4 | The short's topic |
| `GEMINI_API_KEY` | Colab Secrets | Get at [ai.google.dev](https://ai.google.dev) |
| `HF_TOKEN` | Colab Secrets | Optional, needed for gated HF models |
| `YOUTUBE_CLIENT_SECRET` | Colab Secrets | Optional, only for upload |

## Final output
An MP4 saved to `DRIVE_BASE_DIR/final/` + optional YouTube upload.

> **Daily use workflow:** Edit only `TOPIC_PROMPT` in Cell 4, then **Runtime → Run All**.


In [15]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 2 — Install Dependencies               ║
# ╚══════════════════════════════════════════════╝
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

def _apt(*pkgs):
    subprocess.run(["apt-get", "install", "-y", "-q", *pkgs], check=False)

print("📦 Installing system packages...")
_apt("ffmpeg", "imagemagick")

print("📦 Installing Python packages...")
_pip("edge-tts==6.1.9")
_pip("moviepy==1.0.3")
_pip("openai-whisper")
_pip("google-generativeai>=0.7.0")
_pip("diffusers==0.27.2")
_pip("transformers==4.40.0")
_pip("accelerate==0.30.0")
_pip("safetensors")
_pip("google-auth-oauthlib", "google-api-python-client")
_pip("imageio[ffmpeg]", "imageio")
_pip("Pillow>=10.0.0")
_pip("nest_asyncio")
_pip("xformers", "--extra-index-url", "https://download.pytorch.org/whl/cu118")

# Allow ImageMagick policy needed for MoviePy TextClip
_policy = "/etc/ImageMagick-6/policy.xml"
if os.path.isfile(_policy):
    subprocess.run(
        ["sed", "-i", 's/rights="none" pattern="PDF"/rights="read|write" pattern="PDF"/g', _policy],
        check=False
    )
    subprocess.run(
        ["sed", "-i", 's/<policy domain="path" rights="none" pattern="@\*"\/>/<policy domain="path" rights="read|write" pattern="@*"\/>/g', _policy],
        check=False
    )

print("✅ All dependencies installed")




  ["sed", "-i", 's/<policy domain="path" rights="none" pattern="@\*"\/>/<policy domain="path" rights="read|write" pattern="@*"\/>/g', _policy],



📦 Installing system packages...
📦 Installing Python packages...
✅ All dependencies installed


In [16]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 3 — Mount Google Drive & Setup Paths   ║
# ╚══════════════════════════════════════════════╝
from google.colab import drive

print("🔗 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)

# Verify Drive is accessible
_drive_root = '/content/drive/MyDrive'
if not os.path.isdir(_drive_root):
    print("❌ Google Drive mount failed — check permissions and retry")
else:
    print(f"✅ Google Drive mounted at /content/drive/MyDrive")
    print("ℹ️  Directory creation happens when run_full_pipeline() is called in Cell 14")
    print("   Paths are controlled by DRIVE_BASE_DIR in Cell 4 (Config)")


🔗 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted at /content/drive/MyDrive
ℹ️  Directory creation happens when run_full_pipeline() is called in Cell 14
   Paths are controlled by DRIVE_BASE_DIR in Cell 4 (Config)


In [17]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 4 — Config  ← EDIT THIS DAILY          ║
# ╚══════════════════════════════════════════════╝

# ── Core topic ────────────────────────────────────────────────────────────
TOPIC_PROMPT = "5 mind-blowing facts about black holes that will leave you speechless"
NICHE        = "space"   # space | psychology | philosophy | AI | history | creepy

# ── Duration & structure ──────────────────────────────────────────────────
TARGET_DURATION_SECONDS = 45
NUM_SCENES              = 5

# ── Visual generation flags ───────────────────────────────────────────────
USE_AI_VIDEO      = True   # Attempt ModelScope text-to-video first
USE_IMAGE_FALLBACK = True  # Fall back to SDXL-Turbo + Ken Burns if video fails

# ── Audio / subtitle flags ────────────────────────────────────────────────
USE_BACKGROUND_MUSIC = False  # Requires a music file at MUSIC_PATH
GENERATE_SUBTITLES   = True

# ── Upload flag ───────────────────────────────────────────────────────────
UPLOAD_TO_YOUTUBE = False     # Set True when credentials are ready

# ── Paths ─────────────────────────────────────────────────────────────────
DRIVE_BASE_DIR = "/content/drive/MyDrive/YTShorts"
OUTPUT_DIR     = "/content/tmp_shorts"
MUSIC_PATH     = ""   # Optional: full path to a background music MP3

# ── Voice ─────────────────────────────────────────────────────────────────
VOICE_NAME = "en-US-GuyNeural"
# Other good options: "en-US-JennyNeural" | "en-GB-RyanNeural" | "en-AU-WilliamNeural"

# ── Model names ───────────────────────────────────────────────────────────
GEMINI_MODEL_NAME = "gemini-2.5-flash"
VIDEO_MODEL_NAME  = "damo-vilab/text-to-video-ms-1.7b"
IMAGE_MODEL_NAME  = "stabilityai/sdxl-turbo"

# ── Output specs ──────────────────────────────────────────────────────────
ASPECT_RATIO = "9:16"
VIDEO_WIDTH  = 1080
VIDEO_HEIGHT = 1920

# ── API Keys (loaded from Colab Secrets, env, or paste here) ──────────────
import os

GEMINI_API_KEY          = os.environ.get("GEMINI_API_KEY", "")
HF_TOKEN                = os.environ.get("HF_TOKEN", "")
YOUTUBE_CLIENT_SECRET   = os.environ.get("YOUTUBE_CLIENT_SECRET", "")

try:
    from google.colab import userdata
    GEMINI_API_KEY        = GEMINI_API_KEY or (userdata.get("GEMINI_API_KEY") or "")
    HF_TOKEN              = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
    YOUTUBE_CLIENT_SECRET = YOUTUBE_CLIENT_SECRET or (userdata.get("YOUTUBE_CLIENT_SECRET") or "")
except Exception:
    pass

# ── Validation messages ───────────────────────────────────────────────────
print("── Config Loaded ──────────────────────────────")
print(f"   Topic   : {TOPIC_PROMPT[:70]}")
print(f"   Niche   : {NICHE}")
print(f"   Duration: {TARGET_DURATION_SECONDS}s  Scenes: {NUM_SCENES}")
print(f"   AI Video: {USE_AI_VIDEO}   Image Fallback: {USE_IMAGE_FALLBACK}")
if GEMINI_API_KEY:
    print("   ✅ GEMINI_API_KEY detected")
else:
    print("   ⚠️  GEMINI_API_KEY missing — add via Colab Secrets (🔑 icon)")
if HF_TOKEN:
    print("   ✅ HF_TOKEN detected")
else:
    print("   ℹ️  HF_TOKEN not set (optional, needed for gated models)")


── Config Loaded ──────────────────────────────
   Topic   : 5 mind-blowing facts about black holes that will leave you speechless
   Niche   : space
   Duration: 45s  Scenes: 5
   AI Video: True   Image Fallback: True
   ✅ GEMINI_API_KEY detected
   ✅ HF_TOKEN detected


In [18]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 5 — Imports & Environment Checks       ║
# ╚══════════════════════════════════════════════╝
import sys, os, gc, time, json, math, logging, asyncio, subprocess, shutil
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Any, Callable

import nest_asyncio
nest_asyncio.apply()

# ── PyTorch ───────────────────────────────────────────────────────────────
try:
    import torch
    _TORCH_VERSION = torch.__version__
    _CUDA = torch.cuda.is_available()
except ImportError:
    torch = None  # type: ignore
    _TORCH_VERSION = "not installed"
    _CUDA = False

# ── Image / Video ─────────────────────────────────────────────────────────
from PIL import Image
import imageio
import numpy as np

# ── MoviePy ───────────────────────────────────────────────────────────────
from moviepy.editor import (
    VideoFileClip, ImageClip, AudioFileClip, CompositeVideoClip,
    concatenate_videoclips, ColorClip, TextClip, CompositeAudioClip
)
import moviepy.video.fx.all as vfx

# ── edge-tts ──────────────────────────────────────────────────────────────
import edge_tts

# ── Gemini ────────────────────────────────────────────────────────────────
import google.generativeai as genai

# ── Logging ───────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("YTShorts")

# ── Print environment summary ─────────────────────────────────────────────
print(f"🐍 Python      : {sys.version.split()[0]}")
print(f"🔥 PyTorch     : {_TORCH_VERSION}")
if _CUDA and torch:
    dev = torch.cuda.get_device_name(0)
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    free   = (torch.cuda.get_device_properties(0).total_memory
              - torch.cuda.memory_allocated(0)) / 1e9
    print(f"🖥️  GPU         : {dev}")
    print(f"💾 VRAM        : {free:.1f} GB free / {total:.1f} GB total")
else:
    print("⚠️  GPU         : No CUDA GPU — pipeline will use CPU image path")

try:
    import moviepy; print(f"🎬 MoviePy     : {moviepy.__version__}")
except Exception: pass
try:
    print(f"🌐 edge-tts    : {edge_tts.__version__}")
except Exception: pass

print("✅ All imports successful")


🐍 Python      : 3.12.13
🔥 PyTorch     : 2.10.0+cu128
🖥️  GPU         : Tesla T4
💾 VRAM        : 15.6 GB free / 15.6 GB total
🎬 MoviePy     : 1.0.3
🌐 edge-tts    : 6.1.9
✅ All imports successful


In [19]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 6 — Helper Functions                   ║
# ╚══════════════════════════════════════════════╝

def ensure_dir(path: str) -> str:
    """Create directory and return path."""
    os.makedirs(path, exist_ok=True)
    return path


def log_step(step: str, status: str = "START", detail: str = "") -> None:
    """Consistent pipeline log output."""
    icons = {"START": "🔄", "OK": "✅", "WARN": "⚠️", "ERROR": "❌", "INFO": "ℹ️"}
    icon = icons.get(status, "•")
    msg  = f"{icon} [{step}]"
    if detail:
        msg += f"  {detail}"
    print(msg)
    logger.info("[%s] %s %s", step, status, detail)


def validate_file(path: str, min_size_bytes: int = 100) -> bool:
    """Return True if file exists and meets minimum size."""
    return bool(path and os.path.isfile(path) and os.path.getsize(path) >= min_size_bytes)


def retry_call(
    fn: Callable,
    args: tuple = (),
    kwargs: Optional[dict] = None,
    retries: int = 3,
    delay: float = 2.0,
    label: str = "operation",
) -> Any:
    """Retry a callable with exponential back-off."""
    kwargs = kwargs or {}
    last_err: Exception = RuntimeError("No attempts made")
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            last_err = exc
            wait = delay * (2 ** (attempt - 1))
            log_step(label, "WARN",
                     f"Attempt {attempt}/{retries} failed: {exc!s:.80} — retry in {wait:.0f}s")
            time.sleep(wait)
    log_step(label, "ERROR", f"All {retries} attempts failed")
    raise last_err


def safe_download(url: str, dest: str, retries: int = 3) -> bool:
    """Download url to dest with retries. Returns True on success."""
    import urllib.request
    for attempt in range(1, retries + 1):
        try:
            urllib.request.urlretrieve(url, dest)
            if validate_file(dest):
                return True
        except Exception as exc:
            log_step("safe_download", "WARN", f"Attempt {attempt}: {exc}")
            time.sleep(2)
    return False


def clear_vram() -> None:
    """Aggressively free GPU and CPU memory."""
    gc.collect()
    if torch and torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()


def resize_to_vertical(
    input_path: str,
    output_path: str,
    width: int = 1080,
    height: int = 1920,
) -> str:
    """Resize/crop a file to target vertical dimensions via ffmpeg."""
    cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-vf",
        f"scale={width}:{height}:force_original_aspect_ratio=increase,"
        f"crop={width}:{height}",
        "-an", output_path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg resize failed:\n{r.stderr[-400:]}")
    return output_path


def apply_ken_burns_motion(
    image_path: str,
    output_path: str,
    duration: float = 4.0,
    fps: int = 24,
    width: int = VIDEO_WIDTH,
    height: int = VIDEO_HEIGHT,
    zoom_start: float = 1.0,
    zoom_end: float = 1.08,
) -> str:
    """
    Animate a still image with a slow zoom (Ken Burns effect) via ffmpeg.
    Returns output_path on success.
    """
    total_frames = max(int(duration * fps), 1)
    zoom_step    = (zoom_end - zoom_start) / total_frames
    zoom_expr    = f"'min(zoom+{zoom_step:.8f},{zoom_end})'"

    vf = (
        f"scale={width * 2}:{height * 2},"
        f"zoompan=z={zoom_expr}:d={total_frames}"
        f":x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)'"
        f":s={width}x{height}:fps={fps}"
    )
    cmd = [
        "ffmpeg", "-y",
        "-loop", "1", "-i", image_path,
        "-vf", vf,
        "-t", str(duration),
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        output_path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Ken Burns failed:\n{r.stderr[-400:]}")
    return output_path


def setup_directories() -> Dict[str, str]:
    """Create all pipeline directories, verify Drive write access."""
    dirs = {
        "base":         DRIVE_BASE_DIR,
        "audio":        os.path.join(DRIVE_BASE_DIR, "audio"),
        "images":       os.path.join(DRIVE_BASE_DIR, "images"),
        "videos":       os.path.join(DRIVE_BASE_DIR, "videos"),
        "subtitles":    os.path.join(DRIVE_BASE_DIR, "subtitles"),
        "final":        os.path.join(DRIVE_BASE_DIR, "final"),
        "logs":         os.path.join(DRIVE_BASE_DIR, "logs"),
        "temp":         OUTPUT_DIR,
        "temp_scenes":  os.path.join(OUTPUT_DIR, "scenes"),
        "temp_audio":   os.path.join(OUTPUT_DIR, "audio"),
        "temp_images":  os.path.join(OUTPUT_DIR, "images"),
        "temp_subs":    os.path.join(OUTPUT_DIR, "subtitles"),
    }
    for path in dirs.values():
        ensure_dir(path)

    test_path = os.path.join(DRIVE_BASE_DIR, ".write_test")
    try:
        Path(test_path).write_text("ok")
        os.remove(test_path)
        log_step("setup_directories", "OK", f"Base: {DRIVE_BASE_DIR}")
    except Exception as exc:
        log_step("setup_directories", "WARN", f"Drive write-test failed: {exc}")

    return dirs


def check_runtime_environment() -> Dict[str, Any]:
    """Report Python, GPU, and CUDA availability."""
    info: Dict[str, Any] = {
        "python_version":  sys.version,
        "cuda_available":  bool(torch and torch.cuda.is_available()),
        "gpu_name":        None,
        "vram_total_gb":   None,
        "vram_free_gb":    None,
    }
    if info["cuda_available"] and torch:
        props = torch.cuda.get_device_properties(0)
        info["gpu_name"]      = torch.cuda.get_device_name(0)
        info["vram_total_gb"] = round(props.total_memory / 1e9, 2)
        info["vram_free_gb"]  = round(
            (props.total_memory - torch.cuda.memory_allocated(0)) / 1e9, 2
        )
    print(f"🐍 Python : {info['python_version'].split()[0]}")
    if info["cuda_available"]:
        print(f"🖥️  GPU    : {info['gpu_name']}")
        print(f"💾 VRAM   : {info['vram_free_gb']:.1f} GB free / {info['vram_total_gb']:.1f} GB total")
    else:
        print("⚠️  No CUDA — AI video disabled, using image+Ken Burns path")
    return info


def fallback_to_image_scene(
    scene: Dict,
    scene_idx: int,
    output_dir: str,
) -> Optional[str]:
    """Generate an image then animate it. Returns animated .mp4 path or None."""
    image_path = generate_ai_image_scene(scene, scene_idx, output_dir)
    if not image_path:
        return None
    video_path = os.path.join(output_dir, f"scene_{scene_idx:02d}_animated.mp4")
    try:
        apply_ken_burns_motion(
            image_path, video_path,
            duration=float(scene.get("duration", 5.0)),
        )
        return video_path if validate_file(video_path) else None
    except Exception as exc:
        log_step(f"fallback_to_image_scene[{scene_idx}]", "ERROR", str(exc))
        return None


print("✅ Helper functions loaded")


✅ Helper functions loaded


In [20]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 7 — Script Generation (Gemini)         ║
# ╚══════════════════════════════════════════════╝

def generate_short_script() -> Optional[Dict]:
    """
    Use Gemini to generate a structured YouTube Shorts script from TOPIC_PROMPT.
    Returns a dict with hook, full_narration, scenes, title_ideas, description,
    tags, and cta — or None on failure.
    """
    if not GEMINI_API_KEY:
        log_step("generate_short_script", "ERROR",
                 "GEMINI_API_KEY not set. Add it via Colab Secrets (🔑 icon, key name: GEMINI_API_KEY)")
        return None

    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(GEMINI_MODEL_NAME)

    scene_dur = round(TARGET_DURATION_SECONDS / NUM_SCENES, 1)
    word_est  = TARGET_DURATION_SECONDS * 2   # ~2 words/second for TTS

    prompt = f"""You are a top-tier YouTube Shorts scriptwriter.
Write a viral Short for:

TOPIC   : {TOPIC_PROMPT}
NICHE   : {NICHE}
DURATION: {TARGET_DURATION_SECONDS} seconds total
SCENES  : {NUM_SCENES} scenes of ~{scene_dur}s each
WORDS   : ~{word_est} total spoken words

Return ONLY valid JSON — no markdown fences, no explanation.
Use this EXACT structure:
{{
  "hook": "Opening line (0-3 s). Shocking question or statement.",
  "full_narration": "Complete TTS-ready narration for all {TARGET_DURATION_SECONDS}s. Short punchy sentences.",
  "scenes": [
    {{
      "scene_number": 1,
      "duration": {scene_dur},
      "narration_segment": "Words spoken during this scene.",
      "visual_prompt": "Cinematic vertical shot: [specific vivid description]. 9:16 format. {NICHE} aesthetic.",
      "fallback_image_prompt": "Detailed illustration prompt if video generation fails.",
      "mood": "awe|tense|mysterious|dramatic|energetic"
    }}
  ],
  "title_ideas": ["Hook-driven title 1", "Curiosity title 2", "Controversial title 3"],
  "description": "2-3 sentence YouTube description with niche keywords.",
  "tags": ["tag1", "tag2", "tag3", "tag4", "tag5", "tag6"],
  "cta": "End-screen call to action (8 words max)."
}}

Rules:
- Hook must make the viewer stop scrolling in under 3 seconds
- Every narration sentence ≤ 15 words (TTS pacing)
- Visual prompts must be vivid, cinematic, specific
- Generate exactly {NUM_SCENES} scenes
- Tags must include at least 2 trending niche tags
"""

    def _call() -> Dict:
        response = model.generate_content(prompt)
        raw = response.text.strip()
        # Strip markdown fences if model adds them
        if raw.startswith("```"):
            parts = raw.split("```")
            raw = parts[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return json.loads(raw.strip())

    try:
        result = retry_call(_call, label="generate_short_script", retries=3, delay=3.0)
        n_scenes = len(result.get("scenes", []))
        log_step("generate_short_script", "OK", f"{n_scenes} scenes, title: {result.get('title_ideas', ['?'])[0][:60]}")
        return result
    except Exception as exc:
        log_step("generate_short_script", "ERROR", str(exc)[:200])
        return None


def build_scene_plan(script: Dict) -> List[Dict]:
    """
    Normalise raw script scenes into a clean scene plan.
    Each dict has: scene_number, duration, narration_segment,
    visual_prompt, fallback_image_prompt, mood.
    """
    if not script or "scenes" not in script:
        log_step("build_scene_plan", "ERROR", "Script has no 'scenes' key")
        return []

    default_dur = TARGET_DURATION_SECONDS / max(NUM_SCENES, 1)
    scenes = []
    for i, raw in enumerate(script["scenes"]):
        scenes.append({
            "scene_number":        raw.get("scene_number", i + 1),
            "duration":            float(raw.get("duration", default_dur)),
            "narration_segment":   raw.get("narration_segment", ""),
            "visual_prompt":       raw.get("visual_prompt", f"Cinematic {NICHE} scene"),
            "fallback_image_prompt": raw.get(
                "fallback_image_prompt",
                raw.get("visual_prompt", f"Cinematic {NICHE} illustration, high quality")
            ),
            "mood":                raw.get("mood", "dramatic"),
            "clip_path":           None,
        })

    total = sum(s["duration"] for s in scenes)
    log_step("build_scene_plan", "OK",
             f"{len(scenes)} scenes  |  total planned duration: {total:.0f}s")
    return scenes


print("✅ Script generation functions loaded")


✅ Script generation functions loaded


In [21]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 8 — Voice Generation (edge-tts)        ║
# ╚══════════════════════════════════════════════╝

async def _async_tts(text: str, voice: str, output_path: str) -> None:
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_path)


def generate_voiceover(
    narration_text: str,
    output_path: Optional[str] = None,
) -> Optional[str]:
    """
    Generate TTS narration MP3 with edge-tts.
    Returns path to the saved MP3, or None on failure.
    """
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, "audio", "narration.mp3")
    ensure_dir(os.path.dirname(output_path))

    if not narration_text.strip():
        log_step("generate_voiceover", "ERROR", "Empty narration — nothing to generate")
        return None

    log_step("generate_voiceover", "START", f"Voice: {VOICE_NAME}  |  ~{len(narration_text.split())} words")

    try:
        asyncio.run(_async_tts(narration_text, VOICE_NAME, output_path))
    except RuntimeError:
        # Already-running event loop (Colab environment)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(_async_tts(narration_text, VOICE_NAME, output_path))

    if not validate_file(output_path, min_size_bytes=1000):
        log_step("generate_voiceover", "ERROR", "MP3 not created or suspiciously small")
        return None

    size_kb = os.path.getsize(output_path) / 1024
    log_step("generate_voiceover", "OK", f"{size_kb:.0f} KB  →  {output_path}")
    return output_path


print("✅ Voiceover function loaded")


✅ Voiceover function loaded


In [22]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 9 — Visual Generation                  ║
# ╚══════════════════════════════════════════════╝

def generate_ai_video_scene(
    scene: Dict,
    scene_idx: int,
    output_dir: str,
) -> Optional[str]:
    """
    Generate a short AI video clip for one scene with ModelScope text-to-video.
    Runs fp16 on GPU; offloads to CPU when possible to save VRAM.
    Returns saved .mp4 path or None on failure.
    """
    if not USE_AI_VIDEO:
        return None
    if not (torch and torch.cuda.is_available()):
        log_step(f"ai_video[{scene_idx}]", "WARN", "No GPU — skipping AI video")
        return None

    output_path = os.path.join(output_dir, f"scene_{scene_idx:02d}_video.mp4")
    log_step(f"ai_video[{scene_idx}]", "START",
             scene["visual_prompt"][:70] + "…")

    try:
        from diffusers import DiffusionPipeline

        dtype  = torch.float16
        device = "cuda"

        pipe = DiffusionPipeline.from_pretrained(
            VIDEO_MODEL_NAME,
            torch_dtype=dtype,
            trust_remote_code=True,
            use_auth_token=HF_TOKEN or None,
        )

        # Memory-saving methods
        if hasattr(pipe, "enable_model_cpu_offload"):
            pipe.enable_model_cpu_offload()
        else:
            pipe = pipe.to(device)
        if hasattr(pipe, "enable_vae_slicing"):
            pipe.enable_vae_slicing()

        duration    = float(scene.get("duration", 4.0))
        num_frames  = min(int(duration * 8), 24)  # 8 fps, cap at 24 frames for VRAM

        output = pipe(
            prompt               = scene["visual_prompt"],
            num_inference_steps  = 25,
            num_frames           = num_frames,
        )
        frames = output.frames[0]   # list of PIL Images

        with imageio.get_writer(output_path, fps=8, format="mp4") as writer:
            for frame in frames:
                arr = np.array(frame) if isinstance(frame, Image.Image) else frame
                writer.append_data(arr)

        del pipe, output, frames
        clear_vram()

        if validate_file(output_path):
            log_step(f"ai_video[{scene_idx}]", "OK", os.path.basename(output_path))
            return output_path
        return None

    except Exception as exc:
        log_step(f"ai_video[{scene_idx}]", "ERROR", str(exc)[:200])
        clear_vram()
        return None


def generate_ai_image_scene(
    scene: Dict,
    scene_idx: int,
    output_dir: str,
) -> Optional[str]:
    """
    Generate a still image for one scene using SDXL-Turbo (fast, low VRAM).
    Falls back to a plain Pillow gradient if the model fails.
    Returns saved image path or None on complete failure.
    """
    output_path = os.path.join(output_dir, f"scene_{scene_idx:02d}_image.png")
    ensure_dir(output_dir)

    log_step(f"ai_image[{scene_idx}]", "START")

    try:
        from diffusers import AutoPipelineForText2Image

        device  = "cuda" if (torch and torch.cuda.is_available()) else "cpu"
        dtype   = torch.float16 if device == "cuda" else torch.float32
        variant = "fp16" if device == "cuda" else None

        pipe = AutoPipelineForText2Image.from_pretrained(
            IMAGE_MODEL_NAME,
            torch_dtype=dtype,
            variant=variant,
            use_auth_token=HF_TOKEN or None,
        ).to(device)

        for method in ("enable_vae_slicing", "enable_vae_tiling"):
            if hasattr(pipe, method):
                getattr(pipe, method)()

        prompt_text = (
            f"cinematic vertical 9:16, {scene.get('fallback_image_prompt', scene['visual_prompt'])}, "
            f"high quality, sharp detail, {NICHE} aesthetic, dark moody lighting"
        )

        result = pipe(
            prompt               = prompt_text,
            num_inference_steps  = 4,    # SDXL-Turbo is designed for 1-4 steps
            guidance_scale       = 0.0,  # CFG off for Turbo
            height               = 1024,
            width                = 576,
        )
        image = result.images[0].resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.LANCZOS)
        image.save(output_path)

        del pipe, result, image
        clear_vram()

        if validate_file(output_path):
            log_step(f"ai_image[{scene_idx}]", "OK")
            return output_path
        return None

    except Exception as exc:
        log_step(f"ai_image[{scene_idx}]", "WARN", f"Diffusers failed: {exc!s:.120}")
        clear_vram()

    # ── Gradient fallback (no model required) ─────────────────────────────
    try:
        log_step(f"ai_image[{scene_idx}]", "INFO", "Using gradient placeholder image")
        palette = [
            ((10, 5, 30),  (40, 20, 80)),
            ((5, 20, 50),  (20, 60, 120)),
            ((30, 5, 20),  (90, 20, 60)),
            ((5, 25, 40),  (15, 80, 100)),
            ((20, 10, 40), (60, 30, 90)),
        ]
        top_c, bot_c = palette[scene_idx % len(palette)]
        img  = Image.new("RGB", (VIDEO_WIDTH, VIDEO_HEIGHT))
        pix  = img.load()
        for y in range(VIDEO_HEIGHT):
            t = y / VIDEO_HEIGHT
            r = int(top_c[0] + t * (bot_c[0] - top_c[0]))
            g = int(top_c[1] + t * (bot_c[1] - top_c[1]))
            b = int(top_c[2] + t * (bot_c[2] - top_c[2]))
            for x in range(VIDEO_WIDTH):
                pix[x, y] = (r, g, b)
        img.save(output_path)
        log_step(f"ai_image[{scene_idx}]", "OK", "Gradient placeholder saved")
        return output_path
    except Exception as exc2:
        log_step(f"ai_image[{scene_idx}]", "ERROR", f"Gradient fallback failed: {exc2}")
        return None


def generate_scene_visuals(
    scenes: List[Dict],
    output_dir: str,
) -> List[Dict]:
    """
    Orchestrate visual generation for every scene.
    Strategy: AI video → AI image + Ken Burns → gradient + Ken Burns.
    Saves each asset immediately; clears VRAM between scenes.
    Returns the scenes list with 'clip_path' populated.
    """
    scenes_dir = ensure_dir(os.path.join(output_dir, "scenes"))
    log_step("generate_scene_visuals", "START", f"{len(scenes)} scenes")

    for scene in scenes:
        idx = scene["scene_number"]
        clip_path: Optional[str] = None

        # 1. Try AI video
        if USE_AI_VIDEO:
            clip_path = generate_ai_video_scene(scene, idx, scenes_dir)

        # 2. Fallback: AI image + Ken Burns animation
        if clip_path is None and USE_IMAGE_FALLBACK:
            log_step(f"scene[{idx}]", "INFO", "→ image+motion fallback")
            clip_path = fallback_to_image_scene(scene, idx, scenes_dir)

        if clip_path and validate_file(clip_path):
            scene["clip_path"] = clip_path
            log_step(f"scene[{idx}]", "OK", os.path.basename(clip_path))
        else:
            scene["clip_path"] = None
            log_step(f"scene[{idx}]", "WARN", "No clip produced — scene skipped in assembly")

        clear_vram()

    valid = sum(1 for s in scenes if s["clip_path"])
    log_step("generate_scene_visuals", "OK", f"{valid}/{len(scenes)} scenes generated")
    return scenes


print("✅ Visual generation functions loaded")


✅ Visual generation functions loaded


In [23]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 10 — Subtitle Generation               ║
# ╚══════════════════════════════════════════════╝

def _seconds_to_srt_time(seconds: float) -> str:
    """Convert float seconds → SRT timestamp HH:MM:SS,mmm."""
    h   = int(seconds // 3600)
    m   = int((seconds % 3600) // 60)
    s   = int(seconds % 60)
    ms  = int(round((seconds % 1) * 1000))
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def _srt_time_to_seconds(t: str) -> float:
    """Convert SRT timestamp → float seconds."""
    t     = t.strip().replace(",", ".")
    parts = t.split(":")
    return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])


def _get_audio_duration(audio_path: str) -> float:
    """Return duration of audio file in seconds via ffprobe."""
    cmd = [
        "ffprobe", "-v", "quiet", "-print_format", "json",
        "-show_streams", audio_path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    try:
        data = json.loads(r.stdout)
        for stream in data.get("streams", []):
            if "duration" in stream:
                return float(stream["duration"])
    except Exception:
        pass
    return float(TARGET_DURATION_SECONDS)


def _create_chunked_subtitles(
    text: str,
    audio_path: str,
    output_srt: str,
    words_per_chunk: int = 4,
) -> Optional[str]:
    """
    Fallback: split text into equal chunks timed across the audio duration.
    """
    log_step("subtitles", "INFO", "Generating time-chunked SRT fallback")
    duration = _get_audio_duration(audio_path)
    words    = text.strip().split()
    chunks   = [words[i:i + words_per_chunk]
                for i in range(0, len(words), words_per_chunk)]
    if not chunks:
        return None

    tpc  = duration / len(chunks)
    lines = []
    for i, chunk in enumerate(chunks):
        start = i * tpc
        end   = min((i + 1) * tpc, duration)
        lines.append(
            f"{i + 1}\n"
            f"{_seconds_to_srt_time(start)} --> {_seconds_to_srt_time(end)}\n"
            f"{' '.join(chunk).upper()}\n"
        )

    Path(output_srt).write_text("\n".join(lines), encoding="utf-8")
    if validate_file(output_srt):
        log_step("subtitles", "OK", f"Chunked SRT: {len(chunks)} captions")
        return output_srt
    return None


def create_subtitles(
    audio_path: str,
    narration_text: str,
    output_srt: Optional[str] = None,
) -> Optional[str]:
    """
    Generate an SRT subtitle file from the narration audio.
    Uses Whisper-tiny if available; falls back to chunked timing.
    Returns path to .srt file or None.
    """
    if not GENERATE_SUBTITLES:
        return None

    if output_srt is None:
        output_srt = os.path.join(OUTPUT_DIR, "subtitles", "narration.srt")
    ensure_dir(os.path.dirname(output_srt))

    log_step("create_subtitles", "START")

    # ── Whisper path ──────────────────────────────────────────────────────
    try:
        import whisper
        log_step("create_subtitles", "INFO", "Transcribing with Whisper-tiny")
        wmodel = whisper.load_model("tiny")
        result = wmodel.transcribe(audio_path, word_timestamps=False)

        srt_lines = []
        for idx, seg in enumerate(result.get("segments", []), start=1):
            srt_lines.append(
                f"{idx}\n"
                f"{_seconds_to_srt_time(seg['start'])} --> {_seconds_to_srt_time(seg['end'])}\n"
                f"{seg['text'].strip().upper()}\n"
            )

        Path(output_srt).write_text("\n".join(srt_lines), encoding="utf-8")
        del wmodel
        clear_vram()

        if validate_file(output_srt):
            log_step("create_subtitles", "OK", f"Whisper SRT: {len(srt_lines)//4} captions")
            return output_srt

    except Exception as exc:
        log_step("create_subtitles", "WARN", f"Whisper failed ({exc!s:.80}) — using fallback")

    # ── Chunked fallback ──────────────────────────────────────────────────
    return _create_chunked_subtitles(narration_text, audio_path, output_srt)


print("✅ Subtitle functions loaded")


✅ Subtitle functions loaded


In [24]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 11 — Video Assembly (MoviePy)          ║
# ╚══════════════════════════════════════════════╝

def _parse_srt(srt_path: str) -> List[Dict]:
    """Parse an SRT file into a list of {start, end, text} dicts."""
    subs    = []
    content = Path(srt_path).read_text(encoding="utf-8")
    for block in content.strip().split("\n\n"):
        lines = block.strip().split("\n")
        if len(lines) < 3:
            continue
        try:
            times = lines[1].split(" --> ")
            subs.append({
                "start": _srt_time_to_seconds(times[0]),
                "end":   _srt_time_to_seconds(times[1]),
                "text":  " ".join(lines[2:]),
            })
        except Exception:
            continue
    return subs


def _burn_subtitles_moviepy(clip, srt_path: str):
    """Overlay subtitle TextClips from SRT onto a MoviePy clip."""
    subtitles = _parse_srt(srt_path)
    overlays  = [clip]

    for sub in subtitles:
        dur = sub["end"] - sub["start"]
        if dur <= 0:
            continue
        try:
            txt = (
                TextClip(
                    sub["text"],
                    fontsize    = 58,
                    color       = "white",
                    stroke_color= "black",
                    stroke_width= 3,
                    font        = "DejaVu-Sans-Bold",
                    method      = "caption",
                    size        = (VIDEO_WIDTH - 100, None),
                    align       = "center",
                )
                .set_start(sub["start"])
                .set_duration(dur)
                .set_position(("center", int(VIDEO_HEIGHT * 0.74)))
            )
            overlays.append(txt)
        except Exception as exc:
            log_step("subtitles", "WARN", f"TextClip failed: {exc!s:.80}")

    return CompositeVideoClip(overlays) if len(overlays) > 1 else clip


def assemble_short_video(
    scenes: List[Dict],
    audio_path: str,
    srt_path: Optional[str],
    output_path: Optional[str] = None,
    music_path: Optional[str] = None,
) -> Optional[str]:
    """
    Combine scene clips into a 9:16 Shorts video.
    Syncs to narration length, overlays subtitles, optionally mixes music.
    Returns output .mp4 path or None on failure.
    """
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, "assembled_short.mp4")
    ensure_dir(os.path.dirname(output_path))

    valid_scenes = [s for s in scenes if s.get("clip_path") and validate_file(s["clip_path"])]
    if not valid_scenes:
        log_step("assemble_short_video", "ERROR", "No valid clips to assemble")
        return None

    log_step("assemble_short_video", "START",
             f"{len(valid_scenes)} clips  |  audio: {os.path.basename(audio_path)}")

    try:
        narration     = AudioFileClip(audio_path)
        total_dur     = narration.duration
        dur_per_scene = total_dur / len(valid_scenes)

        processed: List = []
        for scene in valid_scenes:
            target_dur = float(scene.get("duration", dur_per_scene))
            try:
                clip = VideoFileClip(scene["clip_path"])
            except Exception:
                clip = ImageClip(scene["clip_path"]).set_duration(target_dur)

            # Normalise to exact target dimensions
            clip = clip.resize((VIDEO_WIDTH, VIDEO_HEIGHT))

            # Match duration: trim long, loop short
            if clip.duration > target_dur:
                clip = clip.subclip(0, target_dur)
            elif clip.duration < target_dur - 0.1:
                n = math.ceil(target_dur / clip.duration)
                clip = concatenate_videoclips([clip] * n).subclip(0, target_dur)

            clip = clip.fadein(0.25).fadeout(0.25)
            processed.append(clip)

        final_clip = concatenate_videoclips(processed, method="compose")

        # Trim to exact narration length
        if final_clip.duration > total_dur:
            final_clip = final_clip.subclip(0, total_dur)

        # ── Audio mix ─────────────────────────────────────────────────────
        audio_tracks = [narration]
        if USE_BACKGROUND_MUSIC and music_path and validate_file(music_path):
            try:
                from moviepy.audio.fx.audio_loop import audio_loop
                music = AudioFileClip(music_path).volumex(0.12)
                music = audio_loop(music, duration=total_dur) if music.duration < total_dur                         else music.subclip(0, total_dur)
                audio_tracks.append(music)
                log_step("assemble_short_video", "INFO", "Background music mixed in")
            except Exception as exc:
                log_step("assemble_short_video", "WARN", f"Music mix failed: {exc!s:.80}")

        composite_audio = (
            CompositeAudioClip(audio_tracks)
            if len(audio_tracks) > 1 else audio_tracks[0]
        )
        final_clip = final_clip.set_audio(composite_audio)

        # ── Subtitles ─────────────────────────────────────────────────────
        if srt_path and validate_file(srt_path):
            final_clip = _burn_subtitles_moviepy(final_clip, srt_path)

        # ── Write ─────────────────────────────────────────────────────────
        final_clip.write_videofile(
            output_path,
            fps        = 24,
            codec      = "libx264",
            audio_codec= "aac",
            temp_audiofile = os.path.join(OUTPUT_DIR, "_tmp_audio.m4a"),
            remove_temp    = True,
            verbose    = False,
            logger     = None,
        )

        final_clip.close()
        narration.close()
        for c in processed:
            c.close()

        if validate_file(output_path, min_size_bytes=10_000):
            size_mb = os.path.getsize(output_path) / 1e6
            log_step("assemble_short_video", "OK",
                     f"{size_mb:.1f} MB  →  {output_path}")
            return output_path

        log_step("assemble_short_video", "ERROR", "Output file missing or too small")
        return None

    except Exception as exc:
        log_step("assemble_short_video", "ERROR", str(exc)[:300])
        return None


print("✅ Video assembly functions loaded")


✅ Video assembly functions loaded


In [25]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 12 — Final Render & Drive Save         ║
# ╚══════════════════════════════════════════════╝

def render_final_video(
    assembled_path: str,
    script: Dict,
    output_dir: Optional[str] = None,
) -> Optional[str]:
    """
    Re-encode the assembled video for reliable playback and save to Drive.
    Also writes a JSON sidecar with metadata.
    Returns final .mp4 path or None on failure.
    """
    if output_dir is None:
        output_dir = os.path.join(DRIVE_BASE_DIR, "final")
    ensure_dir(output_dir)

    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    title_raw  = (script.get("title_ideas") or ["short"])[0]
    title_slug = (
        title_raw.lower()
        .replace(" ", "_")
        .translate(str.maketrans("", "", r'\/:*?"<>|'))
    )[:50]

    final_name = f"{timestamp}_{title_slug}.mp4"
    final_path = os.path.join(output_dir, final_name)

    log_step("render_final_video", "START", final_name)

    # ffmpeg re-encode: clean baseline H.264, faststart for web
    cmd = [
        "ffmpeg", "-y", "-i", assembled_path,
        "-c:v", "libx264", "-preset", "medium", "-crf", "22",
        "-vf", f"scale={VIDEO_WIDTH}:{VIDEO_HEIGHT}:force_original_aspect_ratio=disable",
        "-c:a", "aac", "-b:a", "128k",
        "-movflags", "+faststart",
        "-pix_fmt", "yuv420p",
        final_path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        log_step("render_final_video", "WARN",
                 f"ffmpeg encode failed ({r.returncode}), copying source instead")
        shutil.copy2(assembled_path, final_path)

    if not validate_file(final_path, min_size_bytes=20_000):
        log_step("render_final_video", "ERROR", "Final file invalid or too small")
        return None

    size_mb = os.path.getsize(final_path) / 1e6
    log_step("render_final_video", "OK", f"{size_mb:.1f} MB  →  {final_path}")

    # ── Metadata sidecar ──────────────────────────────────────────────────
    meta = {
        "title":       title_raw,
        "description": script.get("description", ""),
        "tags":        script.get("tags", []),
        "cta":         script.get("cta", ""),
        "topic":       TOPIC_PROMPT,
        "niche":       NICHE,
        "rendered_at": timestamp,
        "file":        final_path,
        "size_mb":     round(size_mb, 2),
    }
    meta_path = final_path.replace(".mp4", "_meta.json")
    Path(meta_path).write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    log_step("render_final_video", "INFO", f"Metadata saved: {meta_path}")

    return final_path


print("✅ Render function loaded")


✅ Render function loaded


In [26]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 13 — Optional YouTube Upload           ║
# ╚══════════════════════════════════════════════╝
#
# Requires client_secrets.json from Google Cloud Console:
#   1. Create a project at console.cloud.google.com
#   2. Enable YouTube Data API v3
#   3. Create OAuth 2.0 credentials → download client_secrets.json
#   4. Upload the file to /content/ or paste JSON as YOUTUBE_CLIENT_SECRET secret
#
# The first run will open an OAuth browser window.
# After auth, the video is uploaded as PRIVATE by default.

def upload_to_youtube(
    video_path: str,
    script: Dict,
    privacy_status: str = "private",
) -> bool:
    """
    Upload video to YouTube via Data API v3.
    Returns True on success, False on any failure.
    Video is uploaded as PRIVATE by default — change manually in YouTube Studio.
    """
    if not UPLOAD_TO_YOUTUBE:
        log_step("upload_to_youtube", "INFO",
                 "Upload disabled — set UPLOAD_TO_YOUTUBE=True in Cell 4 to enable")
        return False

    if not validate_file(video_path):
        log_step("upload_to_youtube", "ERROR", f"Video not found: {video_path}")
        return False

    try:
        import google_auth_oauthlib.flow
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload
    except ImportError:
        log_step("upload_to_youtube", "ERROR",
                 "Missing google-api-python-client. Rerun Cell 2.")
        return False

    SCOPES = ["https://www.googleapis.com/auth/youtube.upload"]
    CLIENT_SECRETS_FILE = "/content/client_secrets.json"

    # Write secrets from env if present
    if YOUTUBE_CLIENT_SECRET and not os.path.isfile(CLIENT_SECRETS_FILE):
        try:
            secret_data = json.loads(YOUTUBE_CLIENT_SECRET)
            Path(CLIENT_SECRETS_FILE).write_text(json.dumps(secret_data))
        except Exception as exc:
            log_step("upload_to_youtube", "ERROR",
                     f"Could not parse YOUTUBE_CLIENT_SECRET: {exc}")
            return False

    if not os.path.isfile(CLIENT_SECRETS_FILE):
        log_step("upload_to_youtube", "ERROR",
                 "client_secrets.json not found at /content/. "
                 "Upload it or set YOUTUBE_CLIENT_SECRET in Colab Secrets.")
        return False

    log_step("upload_to_youtube", "START")

    try:
        flow        = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
            CLIENT_SECRETS_FILE, SCOPES
        )
        credentials = flow.run_local_server(port=0)
        youtube     = build("youtube", "v3", credentials=credentials)

        title       = (script.get("title_ideas") or [TOPIC_PROMPT])[0][:100]
        description = (script.get("description", "") + "\n\n#Shorts")[:5000]
        tags        = list(dict.fromkeys(
            (script.get("tags") or []) + ["shorts", "ytshorts", NICHE]
        ))[:500]

        insert_req = youtube.videos().insert(
            part  = "snippet,status",
            body  = {
                "snippet": {
                    "title":           title,
                    "description":     description,
                    "tags":            tags,
                    "categoryId":      "22",
                    "defaultLanguage": "en",
                },
                "status": {
                    "privacyStatus":          privacy_status,
                    "selfDeclaredMadeForKids": False,
                },
            },
            media_body = MediaFileUpload(video_path, chunksize=-1, resumable=True),
        )

        response = None
        while response is None:
            status, response = insert_req.next_chunk()
            if status:
                print(f"   ⬆️  Upload: {int(status.progress() * 100)}%", end="\r")

        video_id = response.get("id", "unknown")
        log_step("upload_to_youtube", "OK",
                 f"https://youtu.be/{video_id}  (status: {privacy_status})")
        return True

    except Exception as exc:
        log_step("upload_to_youtube", "ERROR", str(exc)[:300])
        return False


print("✅ YouTube upload function loaded")


✅ YouTube upload function loaded


In [27]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 14 — ▶ Run Full Pipeline               ║
# ╚══════════════════════════════════════════════╝

def run_full_pipeline() -> Optional[str]:
    """
    Execute the complete pipeline end-to-end:
      topic → script → voiceover → visuals → subtitles
      → assembly → final render → optional upload

    Returns the path to the final video file, or None if the pipeline failed.
    """
    banner = "=" * 62
    print(f"\n{banner}")
    print("🎬  YouTube Shorts Daily Automation Pipeline")
    print(f"    Topic : {TOPIC_PROMPT[:60]}")
    print(f"    Niche : {NICHE}  |  Target: {TARGET_DURATION_SECONDS}s  |  Scenes: {NUM_SCENES}")
    print(f"{banner}\n")

    # ── Step 0: Setup ────────────────────────────────────────────────────
    print("── Step 0 / 7 ─ Environment & Directories")
    dirs = setup_directories()
    env  = check_runtime_environment()

    # Disable AI video if GPU is unavailable
    global USE_AI_VIDEO
    if not env["cuda_available"]:
        USE_AI_VIDEO = False
        log_step("pipeline", "WARN",
                 "No GPU detected — AI video disabled; using image+Ken Burns path")

    # ── Step 1: Script ───────────────────────────────────────────────────
    print("\n── Step 1 / 7 ─ Script Generation (Gemini)")
    script = generate_short_script()
    if not script:
        log_step("pipeline", "ERROR", "Script generation failed — aborting pipeline")
        return None

    scenes = build_scene_plan(script)
    if not scenes:
        log_step("pipeline", "ERROR", "No scenes in plan — aborting pipeline")
        return None

    # ── Step 2: Voiceover ────────────────────────────────────────────────
    print("\n── Step 2 / 7 ─ Voiceover Generation (edge-tts)")
    narration_text = script.get("full_narration") or " ".join(
        s["narration_segment"] for s in scenes
    )
    audio_path = os.path.join(dirs["temp_audio"], "narration.mp3")
    audio_path = generate_voiceover(narration_text, audio_path)
    if not audio_path:
        log_step("pipeline", "ERROR", "Voiceover failed — aborting pipeline")
        return None

    # ── Step 3: Visuals ──────────────────────────────────────────────────
    print("\n── Step 3 / 7 ─ Visual Generation (scene by scene)")
    scenes = generate_scene_visuals(scenes, dirs["temp"])

    valid_scenes = [s for s in scenes if s.get("clip_path")]
    if not valid_scenes:
        log_step("pipeline", "ERROR", "All visual generation failed — aborting")
        return None

    # ── Step 4: Subtitles ────────────────────────────────────────────────
    print("\n── Step 4 / 7 ─ Subtitle Generation")
    srt_path: Optional[str] = None
    if GENERATE_SUBTITLES:
        srt_out  = os.path.join(dirs["temp_subs"], "narration.srt")
        srt_path = create_subtitles(audio_path, narration_text, srt_out)

    # ── Step 5: Assembly ─────────────────────────────────────────────────
    print("\n── Step 5 / 7 ─ Video Assembly (MoviePy)")
    assembled_path = os.path.join(dirs["temp"], "assembled_short.mp4")
    assembled_path = assemble_short_video(
        scenes      = valid_scenes,
        audio_path  = audio_path,
        srt_path    = srt_path,
        output_path = assembled_path,
        music_path  = MUSIC_PATH or None,
    )
    if not assembled_path:
        log_step("pipeline", "ERROR", "Assembly failed — aborting")
        return None

    # ── Step 6: Final Render ─────────────────────────────────────────────
    print("\n── Step 6 / 7 ─ Final Render & Drive Save")
    final_path = render_final_video(assembled_path, script, dirs["final"])
    if not final_path:
        log_step("pipeline", "ERROR", "Final render failed")
        return None

    # ── Step 7: Upload ───────────────────────────────────────────────────
    print("\n── Step 7 / 7 ─ YouTube Upload")
    uploaded = upload_to_youtube(final_path, script)

    # ── Summary ──────────────────────────────────────────────────────────
    print(f"\n{banner}")
    print("🎉  Pipeline Complete!")
    print(f"   📹 Final video : {final_path}")
    print(f"   🎵 Audio       : {audio_path}")
    print(f"   📝 Subtitles   : {srt_path or 'N/A'}")
    print(f"   📂 Drive base  : {DRIVE_BASE_DIR}")
    print(f"   🏷️  Title       : {(script.get('title_ideas') or ['N/A'])[0]}")
    print(f"   ☁️  Uploaded    : {'✅ Yes' if uploaded else '❌ No (disabled or failed)'}")
    print(f"{banner}\n")

    return final_path


# ── Execute ───────────────────────────────────────────────────────────────
final_video_path = run_full_pipeline()



🎬  YouTube Shorts Daily Automation Pipeline
    Topic : 5 mind-blowing facts about black holes that will leave you s
    Niche : space  |  Target: 45s  |  Scenes: 5

── Step 0 / 7 ─ Environment & Directories
✅ [setup_directories]  Base: /content/drive/MyDrive/YTShorts
🐍 Python : 3.12.13
🖥️  GPU    : Tesla T4
💾 VRAM   : 15.6 GB free / 15.6 GB total

── Step 1 / 7 ─ Script Generation (Gemini)
✅ [generate_short_script]  5 scenes, title: 🤯 Black Hole Facts That Will SHOCK You!
✅ [build_scene_plan]  5 scenes  |  total planned duration: 45s

── Step 2 / 7 ─ Voiceover Generation (edge-tts)
🔄 [generate_voiceover]  Voice: en-US-GuyNeural  |  ~99 words


WSServerHandshakeError: 403, message='Invalid response status', url='wss://speech.platform.bing.com/consumer/speech/synthesize/readaloud/edge/v1?TrustedClientToken=6A5AA1D4EAFF4E9FB37E23D68491D6F4&ConnectionId=fcf3efa7600d43f98a71e36aa62ef479'